In [ ]:
# ── Setup ──────────────────────────────────────────────
import anthropic
import math
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

# 건축공학 도메인 Tool Use — KDS 41 31 00 RC 부재 구조 검토

이 노트북에서는 KDS 41 31 00 (콘크리트구조 설계기준)에 따라  
RC 부재의 **휨 강도**와 **전단 강도**를 검토하는 도구를 구현합니다.

| 버전 | 내용 |
|------|------|
| v1 | 단일 도구 — 휨 검토 |
| v2 | 다중 도구 — 휨 + 전단 검토 |
| v3 | 종합 — 내장 도구(web_search) 활용 |

---

## 헬퍼 함수 (Helper Functions)

S3_03에서 구현한 헬퍼 함수들을 재사용합니다.

In [ ]:
def add_user_message(messages, content):
    """사용자 메시지를 대화 히스토리에 추가합니다."""
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, response):
    """어시스턴트 메시지를 대화 히스토리에 추가합니다."""
    if isinstance(response, anthropic.types.Message):
        messages.append({"role": "assistant", "content": response.content})
    else:
        messages.append({"role": "assistant", "content": response})


def chat(messages, tools=None, system=None):
    """Claude API를 호출합니다."""
    kwargs = {
        "model": MODEL,
        "max_tokens": 4096,
        "messages": messages,
    }
    if tools:
        kwargs["tools"] = tools
    if system:
        kwargs["system"] = system
    return client.messages.create(**kwargs)


def text_from_message(response):
    """Message 응답에서 텍스트 블록들을 합쳐 반환합니다."""
    texts = []
    for block in response.content:
        if block.type == "text":
            texts.append(block.text)
    return "\n".join(texts)

---

## v1: 단일 도구 — 휨 검토 (Flexural Check)

In [ ]:
def check_flexure(b: float, d: float, fck: float, fy: float, As: float) -> str:
    """RC 부재의 휨 강도를 검토합니다 (KDS 41 31 00)."""
    # 등가 직사각형 응력블록 깊이
    a = (As * fy) / (0.85 * fck * b)
    # 공칭 휨 모멘트
    Mn = As * fy * (d - a / 2) / 1e6  # kN·m
    # 설계 휨 강도
    phi_Mn = 0.85 * Mn  # φ = 0.85
    return f"a = {a:.1f} mm, Mn = {Mn:.1f} kN·m, φMn = {phi_Mn:.1f} kN·m"


# 함수 테스트
print(check_flexure(b=300, d=500, fck=30, fy=400, As=1500))

In [ ]:
# ── 휨 검토 도구 스키마 ──
check_flexure_schema = {
    "name": "check_flexure",
    "description": (
        "Checks the flexural strength of an RC member per KDS 41 31 00. "
        "Calculates equivalent rectangular stress block depth (a), "
        "nominal moment (Mn), and design moment (φMn)."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "b": {
                "type": "number",
                "description": "부재 폭 (mm)",
            },
            "d": {
                "type": "number",
                "description": "유효 깊이 (mm)",
            },
            "fck": {
                "type": "number",
                "description": "콘크리트 설계기준압축강도 (MPa)",
            },
            "fy": {
                "type": "number",
                "description": "철근 항복강도 (MPa)",
            },
            "As": {
                "type": "number",
                "description": "인장철근 단면적 (mm²)",
            },
        },
        "required": ["b", "d", "fck", "fy", "As"],
    },
}

tools_v1 = [check_flexure_schema]

In [ ]:
# ── v1 도구 라우터 및 대화 루프 ──

def run_tool(tool_name, tool_input):
    """도구 이름에 해당하는 함수를 실행합니다."""
    if tool_name == "check_flexure":
        return check_flexure(**tool_input)
    else:
        return f"Error: Unknown tool '{tool_name}'"


def run_tools(response):
    """응답의 모든 tool_use 블록을 실행하고 tool_result 리스트를 반환합니다."""
    tool_results = []
    for block in response.content:
        if block.type != "tool_use":
            continue
        tool_name = block.name
        tool_input = block.input
        print(f"  🔧 Running tool: {tool_name}({tool_input})")
        try:
            result = run_tool(tool_name, tool_input)
            print(f"  ✅ Result: {result}")
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": str(result),
            })
        except Exception as e:
            error_msg = f"Error executing {tool_name}: {str(e)}"
            print(f"  ❌ {error_msg}")
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": error_msg,
                "is_error": True,
            })
    return tool_results


def run_conversation(user_message, tools, system=None, verbose=True):
    """사용자 메시지로 시작하여 도구 사용 완료까지 멀티턴 대화를 실행합니다."""
    messages = []
    add_user_message(messages, user_message)
    if verbose:
        print(f"User: {user_message}")
        print("=" * 60)

    turn = 0
    while True:
        turn += 1
        if verbose:
            print(f"\n--- Turn {turn} ---")

        response = chat(messages, tools=tools, system=system)

        if verbose:
            print(f"stop_reason: {response.stop_reason}")

        if response.stop_reason != "tool_use":
            if verbose:
                print(f"\nClaude: {text_from_message(response)}")
            return response, messages

        add_assistant_message(messages, response)
        tool_results = run_tools(response)
        messages.append({"role": "user", "content": tool_results})

    return response, messages

In [ ]:
# ── v1 테스트 ──
response, messages = run_conversation(
    "b=300mm, d=500mm, fck=30MPa, fy=400MPa, As=1500mm² 부재의 휨 강도를 검토해줘",
    tools=tools_v1,
)

---

## v2: 다중 도구 — 휨 + 전단 검토

In [ ]:
def check_shear(b: float, d: float, fck: float, fy: float, Av: float, s: float) -> str:
    """RC 부재의 전단 강도를 검토합니다 (KDS 41 31 00)."""
    # 콘크리트 전단 강도
    Vc = (1 / 6) * math.sqrt(fck) * b * d / 1000  # kN
    # 전단 보강근 기여
    Vs = (Av * fy * d) / s / 1000  # kN
    # 설계 전단 강도
    Vn = Vc + Vs
    phi_Vn = 0.75 * Vn
    return f"Vc = {Vc:.1f} kN, Vs = {Vs:.1f} kN, φVn = {phi_Vn:.1f} kN"


# 함수 테스트
print(check_shear(b=300, d=500, fck=30, fy=400, Av=142, s=200))

In [ ]:
# ── 전단 검토 도구 스키마 ──
check_shear_schema = {
    "name": "check_shear",
    "description": (
        "Checks the shear strength of an RC member per KDS 41 31 00. "
        "Calculates concrete shear capacity (Vc), stirrup contribution (Vs), "
        "and design shear strength (φVn)."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "b": {
                "type": "number",
                "description": "부재 폭 (mm)",
            },
            "d": {
                "type": "number",
                "description": "유효 깊이 (mm)",
            },
            "fck": {
                "type": "number",
                "description": "콘크리트 설계기준압축강도 (MPa)",
            },
            "fy": {
                "type": "number",
                "description": "철근 항복강도 (MPa)",
            },
            "Av": {
                "type": "number",
                "description": "전단 보강근 단면적 (mm²)",
            },
            "s": {
                "type": "number",
                "description": "전단 보강근 간격 (mm)",
            },
        },
        "required": ["b", "d", "fck", "fy", "Av", "s"],
    },
}

tools_v2 = [check_flexure_schema, check_shear_schema]

In [ ]:
# ── v2 도구 라우터 업데이트 ──

def run_tool(tool_name, tool_input):
    """도구 이름에 해당하는 함수를 실행합니다 (v2: 휨 + 전단)."""
    if tool_name == "check_flexure":
        return check_flexure(**tool_input)
    elif tool_name == "check_shear":
        return check_shear(**tool_input)
    else:
        return f"Error: Unknown tool '{tool_name}'"

In [ ]:
# ── v2 테스트: 휨 + 전단 동시 검토 ──
response, messages = run_conversation(
    "b=300mm, d=500mm, fck=30MPa, fy=400MPa, As=1500mm², Av=142mm², s=200mm "
    "부재의 휨 강도와 전단 강도를 모두 검토해줘",
    tools=tools_v2,
)

---

## v3: 종합 — 내장 도구 활용 (web_search)

KDS 기준값을 웹 검색으로 확인하면서 종합 검토 보고서를 생성합니다.  
`web_search`는 Claude 서버에서 직접 실행하는 내장 도구이므로,  
`run_tool`에서 별도로 처리할 필요가 없습니다.

In [ ]:
# ── v3: 시스템 프롬프트 + 도구 목록 ──

system_prompt = (
    "당신은 건축구조 전문가입니다. "
    "KDS 41 31 00 기준에 따라 RC 부재를 검토합니다. "
    "검토 결과를 한국어로 작성하세요."
)

web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 3,
}

tools_v3 = [check_flexure_schema, check_shear_schema, web_search_schema]

In [ ]:
# ── v3 대화 루프 (내장 도구 호환) ──

def run_conversation_v3(user_message, tools, system=None, verbose=True):
    """
    사용자 도구 + 내장 도구를 모두 지원하는 멀티턴 대화 루프.
    내장 도구(web_search 등)는 Claude 서버에서 자동 실행되므로,
    stop_reason이 'tool_use'일 때만 사용자 도구를 처리합니다.
    """
    messages = []
    add_user_message(messages, user_message)
    if verbose:
        print(f"User: {user_message}")
        print("=" * 60)

    turn = 0
    while True:
        turn += 1
        if verbose:
            print(f"\n--- Turn {turn} ---")

        response = chat(messages, tools=tools, system=system)

        if verbose:
            print(f"stop_reason: {response.stop_reason}")

        # 최종 응답
        if response.stop_reason != "tool_use":
            if verbose:
                print(f"\nClaude: {text_from_message(response)}")
            return response, messages

        # 사용자 정의 도구만 실행 (내장 도구는 서버에서 처리됨)
        add_assistant_message(messages, response)

        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue

            tool_name = block.name
            tool_input = block.input

            if verbose:
                print(f"  🔧 Running tool: {tool_name}({tool_input})")

            try:
                result = run_tool(tool_name, tool_input)
                if verbose:
                    print(f"  ✅ Result: {result}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(result),
                })
            except Exception as e:
                error_msg = f"Error executing {tool_name}: {str(e)}"
                if verbose:
                    print(f"  ❌ {error_msg}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": error_msg,
                    "is_error": True,
                })

        messages.append({"role": "user", "content": tool_results})

    return response, messages

In [ ]:
# ── v3 테스트: 종합 구조 검토 ──
response, messages = run_conversation_v3(
    "KDS 41 31 00 기준으로 b=350, d=600, fck=27, fy=400, As=2000, Av=142, s=200 "
    "부재를 종합 검토하고 기준값과 비교해줘",
    tools=tools_v3,
    system=system_prompt,
)